# PRÁCTICA 1: LECTURA Y ALMACENAMIENTO DE ARCHIVOS EN MÚLTIPLES FORMATOS

En esta primera práctica del curso usaremos lo aprendido en las lecciones anteriores para realizar la lectura y almacenamiento de archivos en diferentes formatos.

En particular veremos cómo tomar un archivo JSON (disponible en Internet), cómo procesarlo a través de diccionarios en Python y cómo almacenar el resultado de este procesamiento en un archivo de texto y en un archivo tipo CSV (*comma separated values*).

## Planteamiento del problema

La empresa [City Bike de New York](https://citibikenyc.com/) ofrece el servicio de alquiler de bicicletas para desplazarse por la ciudad.

El sistema de City Bike permite fácilmente al usuario encontrar una bicicleta disponible y cerca a su ubicación, usando un sistema georeferenciado.

Pero en el fondo estos datos georeferenciados son almacenados en archivos JSON. Un ejemplo de este archivo se encuentra en [este enlace](https://feeds.citibikenyc.com/stations/stations.json).

Si tomamos un registro de este archivo, encontraremos estos campos:

- *id* y *stationName*: número identificador y nombre de cada estación
- *availableDocks*: número de puestos disponibles para el parqueo de bicicletas
- *totalDocks*: número total de puestos para el parqueo de bicicletas
- *latitude*, *longitud*: coordenadas geográficas de la estación
- *statusValue*, *statusKey*: indican si la estación está funcionando (*In Service*, 1) o no. Para el caso de este archivo **todas** las estaciones están prestando servicio
- *availableBikes*: cantidad de bicicletas disponibles
- *stAddress1*, *stAddress2*: dirección donde se encuentra ubicada la estación
- *testStation*: indica si se trata de una estación de prueba. En todos los casos este valor es *false*
- *lastCommunicationTime*: fecha y hora en la que se obtuvo la última actualización de la estación
- *city*, *postalCode*, *location*, *altitude*, *landMark*: campos no usados

Supongamos que para una cierta aplicación que queremos desarrollar nos interesa conocer el listado actualizado de estaciones que tengan bicicletas disponibles, y para dichas generar un archivo que contenga: el nombre de la estación, el número de bicicletas disponibles y la dirección.

## Objetivos de la práctica

Implementar un programa capaz de:
1. Leer el archivo JSON directamente desde la web
2. Procesarlo, extrayendo la información relevante para nuestra aplicación
3. Exportar los resultados del procesamiento al formato de archivo más adecuado

## Implementación

Veamos paso a paso la implementación del código para el cumplimiento de cada uno de los objetivos mencionados anteriormente.

### Lectura del archivo JSON

Para leer este archivo usaremos dos módulos incluidos en la Librería Estándar de Python: `urlopen` y `json`.

El módulo `urlopen` nos permitirá acceder desde Python al sitio web (URL) que contiene el archivo JSON y realizar de esta forma la descarga del mismo. Posteriormente usaremos `json` para extraer como tal la información de interés en el formato JSON:

In [ ]:
#import requests, json
from urllib.request import urlopen
import json

# Descargar la información disponible en el sitio web de City Bike NY
#respuesta = urlopen("https://feeds.citibikenyc.com/stations/stations.json")

# Modificación junio 21/2024
# La URL del video original está inactiva. Se actualizó el enlace
# pero el set de datos sigue siendo el mismo
respuesta = urlopen("https://raw.githubusercontent.com/codificandobits/pandas-nivel-basico-data/main/stations.json")
respuesta = respuesta.read()


#respuesta = requests.get("https://feeds.citibikenyc.com/stations/stations.json")

# Imprimamos en pantalla la información que acabamos de descargar (el archivo JSON)
print(respuesta)

In [ ]:
# Para acceder como tal al contenido (es decir al archivo JSON) debemos usar el método
# "text" junto con el módulo "json", generando un diccionario en Python

json_data = json.loads(respuesta)
print(type(json_data))

In [ ]:
# Imprimamos detalles del diccionario obtenido
print(f'json_data - keys: {json_data.keys()}')

In [ ]:
# La información se encuentra en el key "stationBeanList", que es una lista
# de diccionarios
nestaciones = len(json_data['stationBeanList'])
print(f'Cantidad de estaciones: {nestaciones}')
print('\nUn ejemplo de los datos de una estación:')
print(json_data['stationBeanList'][2])

### Procesamiento de los datos y generación de archivo de texto

Crearemos inicialmente un archivo de texto con la información requerida. Por cada diccionario dentro de la lista (es decir por cada estación) extraeremos únicamente:

- Nombre de la estación: el *key* correspondiente es `'stationName'``
- Número de bicicletas disponibles: el *key* correspondiente es `'availableBikes'``
- Dirección: el *key* correspondiente es `'stAddress1'`

Además, para incluir esta información en el archivo de texto verificaremos que haya bicicletas disponibles (el campo `'availableBikes'` debe ser diferente de cero):

In [ ]:
with open('bikesNY.txt','w') as archivo:
    # Crear encabezado
    archivo.write('NOMBRE EST., BIC. DISPONIBLES, DIR.\n')

    # Procesar cada registro (estación), extraer los campos e incluirla
    # en el .txt sólo si hay bicicletas disponibles
    for estacion in json_data['stationBeanList']:
        nombre = estacion['stationName']
        nbicis = estacion['availableBikes']
        direcc = estacion['stAddress1']

        if nbicis != 0:
            archivo.write(f'{nombre}, {nbicis}, {direcc}\n')

¡Y ya tenemos nuestro archivo de texto! Aunque no resulta fácil de leer.

Podemos agregar un formato básico para que se vea organizado por columnas:

In [ ]:
# Definir formato de columnas:
# - {0:<35}: columna 0, alineado a la izquierda (<) y ancho de 35 caracteres
# - {1:>17}: columna 1, alineado a la derecha (>) y ancho de 17 caracteres
# - {2:>35}: columna 2, alineado a la derecha (>) y ancho de 35 caracteres
formato_cols = '{0:<35} {1:>17} {2:>35}'

with open('bikesNY_v2.txt','w') as archivo:
    # Crear encabezado
    archivo.write(formato_cols.format('NOMBRE EST.','BIC. DISPONIBLES', 'DIR.\n'))

    # Procesar cada registro (estación), extraer los campos e incluirla
    # en el .txt sólo si hay bicicletas disponibles
    for estacion in json_data['stationBeanList']:
        nombre = estacion['stationName']
        nbicis = estacion['availableBikes']
        direcc = estacion['stAddress1']

        if nbicis != 0:
            archivo.write(formato_cols.format(nombre, nbicis, direcc+'\n'))

### Procesamiento de los datos y generación de archivo CSV

En el caso anterior vimos que el almacenamiento de datos funciona, pero el formato escogido (TXT) no es el más adecuado pues tenemos datos en formato **tabular**.

Resulta más práctico usar un formato pensado especialmente para datos tabulares. Usaremos el formato CSV (*comma separated values*), muy usado en Ciencia de Datos y que incluso permite abrir el archivo en procesadores de hojas de cálculo como Excel o Google Sheets.

Así que seguiremos una lógica muy similar al anterior pero cambiando el formato, lo que permitirá lograr un resultado equivalente pero con un código más compacto y más fácil de entender:

In [ ]:
import csv  # Módulo que hace parte de la librería estándar de Python

with open('bikesNY.csv', 'w') as archivo:
    writer = csv.writer(archivo)

    # Escribir encabezado: una lista
    encabezado = ['NOMBRE EST.','BIC. DISPONIBLES', 'DIR.']
    writer.writerow(encabezado)

    # Y escribir fila por fila
    for estacion in json_data['stationBeanList']:
        # Parse item
        nombre = estacion['stationName']
        nbicis = estacion['availableBikes']
        direcc = estacion['stAddress1']

        if nbicis != 0:
            fila = [nombre, nbicis, direcc]
            writer.writerow(fila)

El resultado es equivalente pero el código es más entendible. Además podemos abrir el archivo generado en un editor de hojas de cálculo y verlo en el formato tabular.